# Support Integrity Auditor (SIA): pseudo-labeling walkthrough
1. Look at how priority and resolution time relate in the raw data.
2. Show why resolution time cannot serve as a severity signal on its own.
3. Label tickets with the phrase-level severity rubric (`sia/rubric.py`) and inspect the evidence it produces.


In [1]:
import pandas as pd
from sia.labeling import generate_labels, resolution_signal_report, keyword_filler_hit_rate
from sia import rubric

df = pd.read_csv('data/customer_support_tickets.csv')
df.groupby('Priority_Level')['Resolution_Time_Hours'].describe()

,count,mean,std,min,25%,50%,75%,max
Priority_Level,,,,,,,,
Critical,1298.0,12.068567,11.383106,1.0,4.0,9.0,16.0,80.0
High,3416.0,24.520492,23.244518,1.0,7.0,17.0,35.0,120.0
Low,7716.0,45.168351,36.649743,1.0,15.0,34.0,68.0,120.0
Medium,7570.0,44.472919,36.812959,1.0,14.0,33.0,67.0,120.0


## Resolution time follows the *assigned* priority
Pooled over all tickets, fast resolution correlates with High/Critical. That happens because agents resolve
tickets *labelled* urgent first. Within one assigned priority, resolution time barely changes with ticket content,
so it cannot reveal a mislabelled ticket.

In [2]:
labeled = generate_labels(df)
report = resolution_signal_report(labeled)
pd.DataFrame(report['within_priority'])

,assigned_priority,n,median_hours_by_inferred_level,spearman_rho,kruskal_p,eta_squared
0,Low,7716,"{'Low': 34.0, 'Medium': 35.0, 'High': 33.0}",-0.0079,0.4415,0.0
1,Medium,7570,"{'Low': 32.0, 'Medium': 34.0, 'High': 33.0}",-0.0027,0.5481,0.0
2,High,3416,"{'Low': 18.0, 'Medium': 17.0, 'High': 17.0, 'C...",-0.0100,0.4346,0.0
3,Critical,1298,"{'Low': 10.0, 'High': 9.0, 'Critical': 8.0}",-0.0308,0.4635,0.0


## Keyword matching on raw text picks up filler
The descriptions end with random filler words. A plain keyword list (`down`, `manager`, ...) matches that filler.

In [3]:
keyword_filler_hit_rate(df)

{'keyword_hits': 2121, 'hits_in_filler_text': 199, 'filler_share': 0.0938}

## Rubric labels and grounded evidence

In [4]:
print(labeled['mismatch_type'].value_counts())
pd.crosstab(labeled['Priority_Level'], labeled['inferred_severity'])

mismatch_type
Consistent       14745
Hidden Crisis     3575
False Alarm       1680
Name: count, dtype: int64


inferred_severity,Critical,High,Low,Medium
Priority_Level,,,,
Critical,716,492,90,0
High,324,1502,1012,578
Low,0,1281,3639,2796
Medium,0,2294,3105,2171


In [5]:
a = rubric.assess(df.loc[1, 'Ticket_Description'])
print(df.loc[1, 'Ticket_Description'])
rubric.evidence(a)

Hi Support, The application crashes every time I open the settings tab. Speech wall six hour book.


[{'signal': 'text_cue',
  'cue': 'crash',
  'field': 'description',
  'matched_text': 'crashes',
  'indicates': 'High',
  'meaning': 'Application crashes or freezes'}]